# LangChain Prompting – Setup

**Goal:** Run simple prompts using the Groq chat model.

### 1. Install packages
Run this cell once in your notebook. If you install packages while Jupyter is open, restart the kernel if imports fail.

In [ ]:
%pip install -q -U langchain langchain-core langchain-groq python-dotenv

### 2. Configure your Groq API key with a `.env` file

For local development, use `python-dotenv` to load the API key from a `.env` file. This keeps credentials out of notebook code and makes the setup repeatable.

1. In the same project folder as these notebooks, create a file named `.env`.
2. Add this line, replacing the placeholder with your own Groq API key:

   `GROQ_API_KEY=your_actual_groq_api_key`

3. Keep `.env` private. The included `.gitignore` excludes it from Git.
4. Run the next setup cell before initializing the model.

Do not commit, upload, or share your real `.env` file. The included `.env.example` is only a template and contains no real credential.


In [1]:
# Load credentials from the local .env file.
import os
from dotenv import load_dotenv

load_dotenv()  # Reads .env from the current project directory (or parent directories).

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# if not GROQ_API_KEY:
#     raise ValueError(
#         "GROQ_API_KEY was not found. Create a .env file beside the notebooks "
#         "with: GROQ_API_KEY=your_actual_groq_api_key"
#     )

print("Groq API key loaded successfully.")


Groq API key loaded successfully.


### 3. Create the chat model

`ChatGroq` connects LangChain to a Groq-hosted chat model. Model names can change, so use a model currently available in your Groq account.

In [2]:
from langchain_groq import ChatGroq

# Keep the model name configurable because provider model availability can change.
MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    api_key=GROQ_API_KEY
)
print("Groq chat model initialized.")


Groq chat model initialized.


> If Groq reports that the model name is unavailable or decommissioned, replace `model` with a currently supported chat model from your Groq Console.

> **Security and setup:** The original hardcoded API keys have been removed. Run the secure key-entry cell and model setup cell before running model examples. Use a newly generated key; revoke any key that was previously exposed. The model name shown is the one used in the source notebook; change `MODEL_NAME` if it is no longer available in your Groq account.

# Day 5 – Output Parsers and LCEL

## Learning objectives
- Understand why output parsers are useful.
- Request JSON-style structured answers.
- Use StrOutputParser.
- Build chains with LCEL.
- Understand RunnablePassthrough and RunnableParallel.

## 1. What is an output parser?
A parser converts a model response into a more useful format, such as a string or a validated Python object.

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    "Explain {topic} in two simple sentences."
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "SQL"})

print(result)

SQL (Structured Query Language) is a standardized programming language used to create, read, update, and delete data stored in relational databases. It lets you write commands like SELECT, INSERT, UPDATE, and DELETE to manage and retrieve information efficiently.


## 2. JSON output with PydanticOutputParser

In [4]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class StudentInfo(BaseModel):
    name: str = Field(description="Student name")
    course: str = Field(description="Course name")
    duration_months: int = Field(description="Course duration in months")

parser = PydanticOutputParser(pydantic_object=StudentInfo)

prompt = PromptTemplate.from_template(
    """
    Create sample course-enrolment information.
    {format_instructions}
    """
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
student = chain.invoke({})
print(student)
print(type(student))
print(student.name)

name='Alice Johnson' course='Data Science 101' duration_months=6
<class '__main__.StudentInfo'>
Alice Johnson


The model may occasionally return invalid data. A parser can raise an error if the response does not match the required structure. For production applications, handle parsing errors and validate important data.

## 3. LCEL – prompt | model | parser

In [5]:
prompt = PromptTemplate.from_template(
    "Give three benefits of learning {topic}."
)

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"topic": "Docker"})
print(answer)

Here are three key benefits you’ll gain by learning Docker:

| # | Benefit | Why it matters |
|---|---------|----------------|
| **1** | **Consistent, reproducible environments** | Docker packages an application together with all its dependencies (libraries, runtime, OS settings) into a single container image. That image runs the same way on any host that supports Docker—whether it’s a developer’s laptop, a CI server, or a production server—eliminating the classic “it works on my machine” problem. |
| **2** | **Faster development and deployment cycles** | Containers start in seconds, can be built automatically from a Dockerfile, and can be version‑controlled just like code. This lets you spin up test environments, run integration tests, and roll out new releases with minimal manual setup, dramatically shortening the feedback loop. |
| **3** | **Improved resource utilization and scalability** | Docker shares the host kernel, so multiple containers can run side‑by‑side with far less over

## 4. RunnablePassthrough

In [6]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"topic": RunnablePassthrough()}
    | PromptTemplate.from_template("Explain {topic} in simple English.")
    | llm
    | StrOutputParser()
)

print(chain.invoke("Kubernetes"))

**Kubernetes (pronounced “koo‑ber‑net‑ees”) is a tool that helps you run lots of small programs—called *containers*—on many computers, and makes sure they keep working correctly.**  

Think of it like a **traffic controller for a fleet of delivery trucks**:

| Real‑world thing | Kubernetes term |
|------------------|-----------------|
| The whole delivery company | **Cluster** – a group of computers that work together |
| The headquarters that decides where trucks go | **Control plane** (or “master”) – the brain that tells everything what to do |
| Each delivery truck | **Node** – a single computer that actually runs the work |
| A package inside a truck | **Container** – a tiny, self‑contained piece of software |
| A set of packages that belong together (e.g., all the pieces of a web app) | **Pod** – the smallest unit Kubernetes manages; usually one or a few containers that share the same network and storage |
| The plan for how many trucks you need and where they should be | **Deploy

## 5. RunnableParallel

In [7]:
from langchain_core.runnables import RunnableParallel

prompt_a = PromptTemplate.from_template("Explain {topic} in one sentence.")
prompt_b = PromptTemplate.from_template("Give one real-world use of {topic}.")

parallel_chain = RunnableParallel(
    explanation=prompt_a | llm | StrOutputParser(),
    use_case=prompt_b | llm | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "Amazon S3"})

print("EXPLANATION:")
print(result["explanation"])

print("\nUSE CASE:")
print(result["use_case"])

EXPLANATION:
Amazon S3 (Simple Storage Service) is a scalable, durable, and secure cloud object storage service that lets you store and retrieve any amount of data anytime from anywhere over the internet.

USE CASE:
**Use case: Storing and serving static assets for a large‑scale web or mobile application**

A popular real‑world pattern is to place all static files—images, videos, CSS, JavaScript bundles, PDFs, etc.—in an Amazon S3 bucket and serve them directly to users (often via Amazon CloudFront as a CDN).  

**Why it works well**

| Benefit | How S3 provides it |
|---------|--------------------|
| **Scalability** | S3 automatically scales to petabytes and billions of objects without any capacity planning. |
| **High durability & availability** | 99.999999999 % (11 9’s) durability and 99.9 %+ availability across multiple AZs. |
| **Low latency & global reach** | When paired with CloudFront, content is cached at edge locations worldwide, delivering sub‑second load times. |
| **Cost‑e

## 6. Prompt chaining

In [8]:
first_prompt = PromptTemplate.from_template(
    "Give a beginner-friendly definition of {topic}."
)

second_prompt = PromptTemplate.from_template(
    "Create a short quiz question based on this explanation:\n{explanation}"
)

first_chain = first_prompt | llm | StrOutputParser()
second_chain = second_prompt | llm | StrOutputParser()

explanation = first_chain.invoke({"topic": "Python dictionaries"})
quiz = second_chain.invoke({"explanation": explanation})

print("EXPLANATION:")
print(explanation)
print("\nQUIZ:")
print(quiz)

EXPLANATION:
**Python dictionaries – a beginner‑friendly definition**

A *dictionary* in Python is a built‑in data type that stores **pairs of items**: a **key** and a **value**.  
Think of it like a real‑world dictionary (the book) where you look up a **word** (the key) to find its **definition** (the value). In code, you “look up” a key to get the associated value.

### Key points

| Feature | What it means for you |
|---------|-----------------------|
| **Key‑value pair** | Each entry has a unique *key* (like a label) and a *value* (the data you want to keep). |
| **Fast lookup** | Python can find the value for a given key almost instantly, no matter how many pairs there are. |
| **Mutable** | You can add, change, or delete pairs after the dictionary is created. |
| **Unordered (as of Python 3.7, insertion order is preserved)** | The pairs don’t have to be in any particular order, but Python remembers the order you added them. |
| **Keys must be immutable** | Keys can be strings, nu

## 7. Final mini-project – IT course assistant

The assistant accepts a course name and learner level, then generates a short learning plan.

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

course_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful IT course mentor. Keep the plan practical."),
    ("human", """
    Course: {course}
    Student level: {level}

    Create a 5-day beginner learning plan.
    Return:
    - Day number
    - Topic
    - One practical exercise
    """)
])

course_chain = course_prompt | llm | StrOutputParser()

plan = course_chain.invoke({
    "course": "Python",
    "level": "Beginner"
})

print(plan)

**5‑Day Beginner Python Learning Plan**

| Day | Topic | Practical Exercise |
|-----|-------|---------------------|
| **Day 1** | **Python Basics – Syntax, Variables & Data Types** <br>• Installing Python & running scripts <br>• Numbers, strings, booleans <br>• Naming rules & basic operators | **Exercise:** Write a script that asks the user for their name and age, then prints a greeting that includes the name and tells them how many years they have until they turn 100. |
| **Day 2** | **Control Flow – Conditionals & Loops** <br>• `if`, `elif`, `else` <br>• Comparison & logical operators <br>• `while` and `for` loops (range) | **Exercise:** Create a “guess the number” game. The program picks a random integer 1‑50; the user guesses until they get it right, and the program tells them if each guess is too high or too low. |
| **Day 3** | **Collections – Lists & Dictionaries** <br>• Creating, indexing, slicing lists <br>• Adding/removing items <br>• Key‑value pairs in dictionaries <br>• Loo

## Final revision checklist

- [ ] PromptTemplate
- [ ] ChatPromptTemplate
- [ ] System, Human, and AI messages
- [ ] MessagesPlaceholder
- [ ] Zero-shot, one-shot, and few-shot
- [ ] FewShotPromptTemplate
- [ ] Example selectors
- [ ] Structured output and PydanticOutputParser
- [ ] StrOutputParser
- [ ] LCEL
- [ ] RunnablePassthrough
- [ ] RunnableParallel
- [ ] Prompt chaining

**Trainer note:** Run the setup cells at the beginning of each notebook. Keep the Groq API key private. Model responses can vary between runs, even when the prompt is unchanged.

## 8. JSON output: structured data for applications

JSON is useful when another program must read the model response. A prompt that says "return JSON" is not enough by itself; parse and validate the result before using it.

**Why it is useful:** Applications can reliably access fields such as `category`, `priority`, or `summary` instead of trying to interpret free-form prose.

In [10]:
import json
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_prompt = PromptTemplate.from_template(
    """Extract a support ticket into JSON.
{format_instructions}
Ticket: {ticket}
"""
).partial(format_instructions=json_parser.get_format_instructions())

json_chain = json_prompt | llm | json_parser
ticket_data = json_chain.invoke({
    "ticket": "The production database is unavailable. This is urgent."
})
print(ticket_data)
print("Python type:", type(ticket_data))

{'title': 'Production database unavailable', 'description': 'The production database is unavailable. This is urgent.', 'category': 'Database', 'priority': 'High', 'urgency': 'Urgent', 'status': 'Open'}
Python type: <class 'dict'>


## 9. Pydantic validation and type safety

`PydanticOutputParser` parses model output into a Pydantic model. Field descriptions guide the model; Python types and validators help validate the parsed result.

**Why it is useful:** Your application can work with typed attributes and reject invalid values before downstream processing. Always handle parsing and validation errors.

In [11]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class SupportTicket(BaseModel):
    category: str = Field(description="Technical, Billing, or Account")
    priority: str = Field(description="Low, Medium, or High")
    summary: str = Field(description="One-sentence summary")

support_parser = PydanticOutputParser(pydantic_object=SupportTicket)
support_prompt = PromptTemplate.from_template(
    """Analyze this support ticket.
{format_instructions}
Ticket: {ticket}
"""
).partial(format_instructions=support_parser.get_format_instructions())

support_chain = support_prompt | llm | support_parser
parsed_ticket = support_chain.invoke({
    "ticket": "I cannot log in after resetting my password. I need access today."
})
print(parsed_ticket)
print("Category:", parsed_ticket.category)
print("Priority:", parsed_ticket.priority)

category='Account' priority='High' summary='User cannot log in after resetting password and needs immediate access.'
Category: Account
Priority: High


## 10. RunnableLambda: add ordinary Python logic to a chain

`RunnableLambda` wraps a Python function so it can participate in an LCEL pipeline. Use it for simple transformations, validation, formatting, or business logic.

**Why it is useful:** You can combine model calls with deterministic Python steps. Keep critical validation in code rather than asking the model to enforce it.

In [12]:
from langchain_core.runnables import RunnableLambda

def normalize_topic(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("Pass a non-empty topic string")
    return {"topic": value.strip().title()}

normalize = RunnableLambda(normalize_topic)
normalization_chain = (
    normalize
    | PromptTemplate.from_template("Explain {topic} in one sentence.")
    | llm
    | StrOutputParser()
)
print(normalization_chain.invoke("python decorators"))

A Python decorator is a higher‑order function that takes another function (or method) as input, adds or modifies its behavior, and returns a new callable—typically applied with the `@decorator_name` syntax above the function definition.


## 11. Batch and streaming

- `batch()` runs a chain for multiple inputs. Providers may process requests concurrently depending on configuration and limits.
- `stream()` yields output chunks as they become available. Chunks are not necessarily complete sentences or valid JSON until the stream finishes.

**Why it is useful:** Batch helps process lists of tasks; streaming can make interactive applications feel more responsive.

In [13]:
simple_chain = (
    PromptTemplate.from_template("Define {topic} in one short sentence.")
    | llm
    | StrOutputParser()
)

batch_results = simple_chain.batch([
    {"topic": "API"},
    {"topic": "Docker"},
    {"topic": "Kubernetes"}
])
for item in batch_results:
    print(item)

print("\\nStreaming response:")
for chunk in simple_chain.stream({"topic": "Terraform"}):
    print(chunk, end="")
print()

An API (Application Programming Interface) is a set of rules and protocols that lets different software applications communicate and use each other's functionality.
Docker is a platform that packages applications and their dependencies into lightweight, portable containers for consistent execution across any environment.
Kubernetes is an open‑source platform that automates the deployment, scaling, and management of containerized applications across clusters of machines.
\nStreaming response:
Terraform is an open‑source infrastructure‑as‑code tool that lets you declaratively provision and manage cloud and on‑premises resources using a unified configuration language.


## 12. Error handling around model and parser calls

Model calls can fail because of network problems, rate limits, invalid model names, authentication failures, or malformed output. Production code should catch expected exceptions, log useful diagnostics without exposing API keys, and decide whether a retry is safe.

Do not retry every error blindly: authentication and invalid-request errors generally need configuration changes, not repeated requests.

In [14]:
from langchain_core.output_parsers import StrOutputParser

safe_chain = (
    PromptTemplate.from_template("Explain {topic} in one sentence.")
    | llm
    | StrOutputParser()
)

try:
    print(safe_chain.invoke({"topic": "CI/CD"}))
except Exception as exc:
    print("Chain failed:", type(exc).__name__)
    print("Check credentials, model availability, network, and provider limits.")

CI/CD (Continuous Integration and Continuous Delivery/Deployment) is an automated workflow that continuously merges code changes into a shared repository, runs tests, and then automatically builds, packages, and releases the software to production or staging environments.


## 13. Optional next-level topics to study after these five days

These are valuable extensions and are not required to run the core notebooks:

- **Tool calling:** let a model request a narrowly scoped tool; validate arguments and enforce permissions in application code.
- **Retrieval-Augmented Generation (RAG):** retrieve relevant documents and place grounded excerpts into the prompt.
- **Conversation memory/state:** store selected conversation state outside the prompt; define retention and privacy rules.
- **Prompt versioning:** keep prompts in source control and record which version produced each result.
- **Observability and tracing:** inspect inputs, outputs, latency, token usage, and failures. Avoid logging secrets or unnecessary personal data.
- **Model comparison:** run the same evaluation set against different models and prompt versions.

**Final mini-project:** Build an IT course assistant that accepts a course and learner level, generates a structured 5-day plan, validates the output with Pydantic, and creates three quiz questions from the plan.

## Day 5 completion checklist

- [ ] I can explain the concept without reading the notebook.
- [ ] I understand why and when it is useful.
- [ ] I can modify the example for a different IT topic.
- [ ] I can identify at least one limitation or failure case.
- [ ] I completed the practice task.

**Recommended workflow:** Read the explanation → run the example → change the input → compare outputs → explain the result in your own words.

# Advanced Day 5 – Deep Dive into Output Parsers, LCEL, and Chains

This extension builds on the existing Day 5 lessons. It introduces additional parser types, more LCEL runnable patterns, chain design, branching, validation, and debugging. Run the original setup cells first so `llm` is initialized.

> **Important:** A parser structures or validates a model response; it does not independently prove that the model's content is factually correct. Validate important business rules in Python and verify important facts against trusted sources.

## 14. Output parser types – which parser should you choose?

| Parser | Output | When to use |
|---|---|---|
| `StrOutputParser` | Python `str` | Plain text, summaries, explanations |
| `JsonOutputParser` | Python dict/list | JSON-like data without a fixed Pydantic class |
| `PydanticOutputParser` | Pydantic model instance | Fixed schema and type validation |
| `CommaSeparatedListOutputParser` | Python list of strings | Simple comma-separated items |
| `MarkdownListOutputParser` | List of strings | Markdown bullet lists |
| `NumberedListOutputParser` | List of strings | Numbered steps |
| `XMLOutputParser` | Parsed XML-like structure | XML-oriented integrations; inspect its output shape |

**Selection rule:** Use plain strings for display, JSON for flexible records, and Pydantic when your application needs a defined schema and typed fields.

In [15]:
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    CommaSeparatedListOutputParser,
    MarkdownListOutputParser,
    NumberedListOutputParser,
    XMLOutputParser
)

# A) CommaSeparatedListOutputParser
csv_parser = CommaSeparatedListOutputParser()
csv_prompt = PromptTemplate.from_template(
    "List exactly four common Python data types. {format_instructions}"
).partial(format_instructions=csv_parser.get_format_instructions())

csv_chain = csv_prompt | llm | csv_parser
data_types = csv_chain.invoke({})
print(data_types)
print(type(data_types))

# B) MarkdownListOutputParser
markdown_parser = MarkdownListOutputParser()
markdown_prompt = PromptTemplate.from_template(
    "Give three benefits of Docker as a Markdown bullet list. {format_instructions}"
).partial(format_instructions=markdown_parser.get_format_instructions())

markdown_chain = markdown_prompt | llm | markdown_parser
print(markdown_chain.invoke({}))

# C) NumberedListOutputParser
numbered_parser = NumberedListOutputParser()
numbered_prompt = PromptTemplate.from_template(
    "Give three steps to create a Python virtual environment. {format_instructions}"
).partial(format_instructions=numbered_parser.get_format_instructions())

numbered_chain = numbered_prompt | llm | numbered_parser
print(numbered_chain.invoke({}))

['int', 'float', 'str', 'list']
<class 'list'>
['Consistent environments across development, testing, and production  ', 'Faster deployment and scaling through lightweight, portable containers  ', 'Efficient resource utilization by sharing the host OS kernel, reducing overhead compared to VMs']
['Open a terminal (or command prompt) and navigate to the project directory where you want the virtual environment.', 'Run the command `python -m venv venv` (you can replace `venv` with any name you prefer for the environment).', 'Activate the environment:  ']


## 15. `parse()` versus `invoke()`

- `parse(text)` parses an already-existing string. It does not call the language model.
- `invoke(input)` runs a runnable. On a parser, it receives the previous step's model output and parses it.
- In a chain such as `prompt | llm | parser`, `invoke()` runs the complete sequence.

**Why it matters:** You can test parser behaviour independently from model behaviour, which makes debugging much easier.

In [16]:
from langchain_core.output_parsers import JsonOutputParser

json_parser_demo = JsonOutputParser()

raw_json_text = '{"course": "Python", "duration_weeks": 6}'
parsed_with_parse = json_parser_demo.parse(raw_json_text)

print("parse() result:", parsed_with_parse)
print("Type:", type(parsed_with_parse))

# invoke() on the parser itself parses the supplied input string.
parsed_with_invoke = json_parser_demo.invoke(raw_json_text)
print("invoke() result:", parsed_with_invoke)

parse() result: {'course': 'Python', 'duration_weeks': 6}
Type: <class 'dict'>
invoke() result: {'course': 'Python', 'duration_weeks': 6}


## 16. JSON parser with a flexible schema

`JsonOutputParser` is useful when the exact fields vary by task. It can also provide format instructions. The application should still check required keys and values after parsing.

**Real-world use:** Extracting a ticket where optional fields may be present, or generating metadata that is consumed by a downstream service.

In [17]:
json_parser = JsonOutputParser()
json_prompt = PromptTemplate.from_template(
    """Extract a ticket record.
{format_instructions}
Use these fields: category, priority, summary.
Ticket: {ticket}
"""
).partial(format_instructions=json_parser.get_format_instructions())

json_chain = json_prompt | llm | json_parser
ticket = json_chain.invoke({
    "ticket": "The application is unavailable for all users. This is urgent."
})

required_keys = {"category", "priority", "summary"}
missing_keys = required_keys - set(ticket.keys())
if missing_keys:
    raise ValueError(f"Missing required keys: {missing_keys}")

print(ticket)
print("Category:", ticket["category"])

{'category': 'Availability', 'priority': 'Urgent', 'summary': 'Application unavailable for all users'}
Category: Availability


## 17. Stronger Pydantic validation

Pydantic field types and constraints can reject malformed output. For example, `ge` and `le` constrain numeric ranges. Use `Literal` to restrict a field to a known set of values.

**Why it is useful:** The application gets a predictable object and can stop invalid values from flowing into later steps.

In [18]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class CourseRecommendation(BaseModel):
    course: str = Field(description="Recommended IT course")
    level: Literal["Beginner", "Intermediate", "Advanced"]
    duration_weeks: int = Field(ge=1, le=52, description="Duration in weeks")
    reasons: list[str] = Field(min_length=2, max_length=4)

course_parser = PydanticOutputParser(pydantic_object=CourseRecommendation)
course_prompt = PromptTemplate.from_template(
    """Recommend a suitable course for a student interested in cloud computing.
{format_instructions}
"""
).partial(format_instructions=course_parser.get_format_instructions())

course_recommendation_chain = course_prompt | llm | course_parser
recommendation = course_recommendation_chain.invoke({})
print(recommendation)
print("Validated duration:", recommendation.duration_weeks)

course='AWS Certified Solutions Architect – Associate' level='Intermediate' duration_weeks=10 reasons=['Covers core AWS services and architectural best practices essential for cloud computing roles.', 'Provides hands‑on labs and real‑world scenarios to build practical deployment skills.', 'Recognized industry certification that enhances employability and career advancement.']
Validated duration: 10


## 18. Native structured output versus output parsers

Some chat-model integrations support `llm.with_structured_output(PydanticModel)`. This asks the provider/model integration to return data in a schema-aware way. It is different from asking for text and then parsing that text with `PydanticOutputParser`.

- **Parser approach:** prompt → model text → parser → object.
- **Native structured-output approach:** model configured with schema → structured result.

Support varies by provider, model, and integration version. If the selected Groq model/integration does not support it, continue using `PydanticOutputParser`.

In [19]:
# Optional: run only if the selected ChatGroq model supports structured output.
# If the provider/model does not support this feature, use the PydanticOutputParser
# approach in the previous section instead.

try:
    structured_llm = llm.with_structured_output(CourseRecommendation)
    native_result = structured_llm.invoke(
        "Recommend a beginner cloud course for a student who knows Python."
    )
    print(native_result)
    print(type(native_result))
except Exception as exc:
    print("Native structured output is not available for this model/configuration.")
    print("Error type:", type(exc).__name__)

course='AWS Cloud Practitioner Essentials' level='Beginner' duration_weeks=6 reasons=['Covers fundamental cloud concepts and AWS services with a focus on real-world use cases', 'Includes hands‑on labs that let you use Python with AWS SDK (boto3) to automate tasks', 'Prepares you for the AWS Certified Cloud Practitioner exam, boosting your resume', 'Designed for learners with programming experience, so Python knowledge accelerates progress']
<class '__main__.CourseRecommendation'>


## 19. LCEL fundamentals – Runnable interface

LCEL means **LangChain Expression Language**. It lets you compose runnable components with the pipe operator `|`.

A typical chain is:

`PromptTemplate → ChatModel → OutputParser`

Each component receives the previous component's output and passes its own output to the next component.

**Why it is useful:** It makes workflows composable, reusable, testable, and supports common operations such as `invoke`, `batch`, and `stream`.

In [20]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template("Explain {topic} in two lines.")
text_chain = prompt | llm | StrOutputParser()

# The input to the chain is a dictionary because the prompt has {topic}.
answer = text_chain.invoke({"topic": "Kubernetes"})
print(answer)

# Inspect the chain's component representation.
print(text_chain)

Kubernetes is an open‑source platform that automates the deployment, scaling, and management of containerized applications across clusters of machines.  
It abstracts the underlying infrastructure, letting you define desired state (e.g., replicas, networking, storage) and continuously reconciles the actual state to match it.
first=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain {topic} in two lines.') middle=[ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 

## 20. `RunnableSequence` – explicit sequential chain

The pipe operator creates a sequence of runnables. You can also construct `RunnableSequence` explicitly.

**Use case:** A task that must happen in order, such as classify a ticket, then write a response using the classification.

In [21]:
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser

classify_prompt = PromptTemplate.from_template(
    "Classify this issue as Technical, Billing, or Account. Return only the category: {issue}"
)
explain_prompt = PromptTemplate.from_template(
    "Write one helpful next step for a customer whose issue category is: {category}"
)

classifier = classify_prompt | llm | StrOutputParser()
next_step = explain_prompt | llm | StrOutputParser()

sequential_chain = RunnableSequence(
    first=classifier,
    last=next_step
)

print(sequential_chain.invoke({"issue": "I was charged twice this month"}))

Please review your most recent invoice, note the specific charge(s) you’re questioning, and reply with the invoice number and details so we can investigate and resolve the billing issue for you.


## 21. `RunnableParallel` – run independent tasks together

`RunnableParallel` sends the same input to multiple branches and returns a dictionary keyed by branch name. Each branch can be a chain.

**Use case:** Generate a definition, example, and quiz question from the same topic independently. Parallel execution can reduce wall-clock time compared with waiting for each independent call one by one; provider rate limits still apply.

In [22]:
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

parallel = RunnableParallel(
    definition=PromptTemplate.from_template("Define {topic} simply.") | llm | StrOutputParser(),
    example=PromptTemplate.from_template("Give one practical example of {topic}.") | llm | StrOutputParser(),
    quiz=PromptTemplate.from_template("Write one quiz question about {topic}.") | llm | StrOutputParser()
)

parallel_result = parallel.invoke({"topic": "AWS IAM"})
for key, value in parallel_result.items():
    print(f"\\n{key.upper()}:\\n{value}")

\nDEFINITION:\n**AWS IAM (Identity and Access Management)** is a service that lets you control who can do what in your Amazon Web Services account.  

- **Identity** – You create users, groups, and roles that represent people, applications, or services.  
- **Access** – You attach policies (permissions) to those identities to specify which AWS resources they can view, modify, or manage.  

In short, IAM is the security gatekeeper that lets you define **who** (identity) can perform **which actions** (access) on **which resources** (AWS services).
\nEXAMPLE:\n**Practical Example: Granting an EC2 Instance Read‑Only Access to a Specific S3 Bucket**

---

### Scenario  
You have a web‑application running on an Amazon EC2 instance that needs to **download configuration files** stored in an S3 bucket called `my‑app‑config`. The instance should **not be able to write, delete, or list any other buckets**. Using IAM you can give the instance exactly the permissions it needs—nothing more.

---

#

## 22. `RunnablePassthrough` and `.assign()`

- `RunnablePassthrough()` forwards the original input unchanged.
- `RunnablePassthrough.assign(...)` adds new keys to an input dictionary while preserving existing keys.

**Why it is useful:** You can enrich data with generated fields without losing the original input. This pattern is common in multi-step workflows.

In [23]:
from langchain_core.runnables import RunnablePassthrough

summary_chain = (
    PromptTemplate.from_template("Summarize this text in one sentence: {text}")
    | llm
    | StrOutputParser()
)

enriched_chain = RunnablePassthrough.assign(
    summary=summary_chain
)

enriched = enriched_chain.invoke({
    "text": "Docker packages an application and its dependencies into containers."
})

print(enriched)
print("Original text still present:", "text" in enriched)
print("Generated summary:", enriched["summary"])

{'text': 'Docker packages an application and its dependencies into containers.', 'summary': 'Docker bundles an application together with all its required dependencies into isolated containers.'}
Original text still present: True
Generated summary: Docker bundles an application together with all its required dependencies into isolated containers.


## 23. `RunnableLambda` – add Python functions to an LCEL chain

A `RunnableLambda` converts a Python function into a runnable. Use it for deterministic transformations, input checks, formatting, or business rules.

**Use case:** Normalize a topic before sending it to the model, or validate a model-generated category against an allowed set.

In [24]:
from langchain_core.runnables import RunnableLambda

def validate_and_normalize(inputs):
    topic = inputs.get("topic", "")
    if not isinstance(topic, str) or not topic.strip():
        raise ValueError("topic must be a non-empty string")
    return {"topic": topic.strip().title()}

normalize_runnable = RunnableLambda(validate_and_normalize)

lambda_chain = (
    normalize_runnable
    | PromptTemplate.from_template("Explain {topic} in one sentence.")
    | llm
    | StrOutputParser()
)

print(lambda_chain.invoke({"topic": "terraform"}))

Terraform is an open‑source infrastructure‑as‑code tool that lets you define, provision, and manage cloud and on‑premises resources across multiple providers using declarative configuration files.


## 24. `RunnableBranch` – conditional chain routing

`RunnableBranch` chooses a branch based on conditions. Conditions are checked in order; the first matching condition is used. The final branch is the default.

**Use case:** Route beginner questions to a beginner explanation prompt and advanced questions to a deeper technical prompt.

In [25]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

beginner_chain = (
    PromptTemplate.from_template("Explain {topic} using simple language and an analogy.")
    | llm | StrOutputParser()
)
advanced_chain = (
    PromptTemplate.from_template("Explain {topic} with architecture details and trade-offs.")
    | llm | StrOutputParser()
)

router = RunnableBranch(
    (lambda x: x["level"].lower() == "beginner", beginner_chain),
    advanced_chain  # default branch for other levels
)

print(router.invoke({"topic": "Kubernetes networking", "level": "beginner"}))
print(router.invoke({"topic": "Kubernetes networking", "level": "advanced"}))

## Kubernetes Networking in Plain English  
*(Think of a tiny, self‑contained city that can grow, shrink, and move around whenever you want.)*

---

### 1. The Big Picture

| Kubernetes term | What it *does* | City‑style analogy |
|-----------------|----------------|--------------------|
| **Pod** | The smallest unit that runs your container(s). It gets its own IP address and can talk to any other pod directly. | A **house**. Every house has its own mailbox (IP) and can call any other house on the street. |
| **Node** | A physical or virtual machine that hosts many pods. | A **city block**. All the houses on the block share the same street address (the node’s IP). |
| **Service** | A stable “virtual IP” that groups a set of pods and load‑balances traffic to them. | A **post office** that knows which houses belong to a particular street (e.g., “all the houses that run the web app”) and forwards mail to the right house, even if the houses move. |
| **ClusterIP** | The internal IP of a Se

## 25. `itemgetter` – pass selected dictionary values

Python's `operator.itemgetter` is often used inside LCEL to select one or more keys from an input dictionary and feed them to a chain.

**Why it is useful:** It makes data routing explicit when a chain needs only a subset of a larger input.

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnableParallel

select_fields = RunnableParallel(
    course=itemgetter("course"),
    level=itemgetter("level")
)

selected = select_fields.invoke({
    "course": "AWS",
    "level": "Beginner",
    "student_name": "Veera"
})
print(selected)

{'course': 'AWS', 'level': 'Beginner'}


## 26. `with_fallbacks()` – define a backup path

A runnable can be configured with fallback runnables. This can help recover from selected runtime failures, but a fallback should not hide broken credentials, invalid inputs, or repeated provider errors.

**Production tip:** Log which path succeeded and keep fallback behaviour transparent.

In [27]:
from langchain_core.runnables import RunnableLambda

primary = RunnableLambda(
    lambda value: (_ for _ in ()).throw(RuntimeError("Simulated primary failure"))
)
backup = RunnableLambda(lambda value: "Backup response: " + str(value))

resilient_chain = primary.with_fallbacks([backup])
print(resilient_chain.invoke("hello"))

Backup response: hello


## 27. `with_retry()` – retry transient failures carefully

`with_retry()` can retry a runnable when it raises an exception. Retries are most useful for transient failures. They do not fix invalid API keys, unsupported model names, or malformed prompt design. Retries can also increase latency and cost.

Use a small retry count and monitor provider limits.

In [28]:
from langchain_core.runnables import RunnableLambda

attempts = {"count": 0}

def sometimes_fails(value):
    attempts["count"] += 1
    if attempts["count"] < 2:
        raise RuntimeError("Temporary simulated issue")
    return "Succeeded on attempt " + str(attempts["count"])

retry_chain = RunnableLambda(sometimes_fails).with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=False
)

print(retry_chain.invoke("start"))

Succeeded on attempt 2


## 28. Batch, stream, and async – common execution methods

| Method | Purpose |
|---|---|
| `invoke()` | One input, one completed result |
| `batch()` | Multiple inputs |
| `stream()` | Yield output chunks as they arrive |
| `ainvoke()` | Asynchronous single invocation |
| `abatch()` | Asynchronous batch |
| `astream()` | Asynchronous streaming |

Streaming chunks may split words or JSON tokens. Do not treat partial output as a complete validated record.

In [29]:
import asyncio

# Synchronous examples
short_chain = (
    PromptTemplate.from_template("Define {topic} in one sentence.")
    | llm
    | StrOutputParser()
)

print("INVOKE:", short_chain.invoke({"topic": "CI/CD"}))
print("BATCH:", short_chain.batch([
    {"topic": "Git"},
    {"topic": "Docker"}
]))

print("STREAM:", end=" ")
for piece in short_chain.stream({"topic": "Terraform"}):
    print(piece, end="")
print()

# Async example. In Jupyter, await can be used directly in a cell.
async_result = await short_chain.ainvoke({"topic": "Kubernetes"})
print("ASYNC:", async_result)

INVOKE: CI/CD (Continuous Integration and Continuous Delivery/Deployment) is a set of automated practices that regularly merge code changes into a shared repository, run comprehensive tests, and swiftly release validated software to production or staging environments.
BATCH: ['Git is a distributed version‑control system that tracks changes to files, enables multiple collaborators to work on a project simultaneously, and allows you to revert, branch, and merge code histories efficiently.', 'Docker is a platform that uses containerization to package an application and all its dependencies into a lightweight, portable, and isolated executable unit that can run consistently across any environment.']
STREAM: Terraform is an open‑source infrastructure‑as‑code tool that lets you declaratively define, provision, and manage cloud and on‑premises resources across multiple providers using a simple, version‑controlled configuration language.
ASYNC: Kubernetes is an open‑source platform that automa

## 29. Parser failures – handle them deliberately

A parser can fail when the model returns invalid JSON, omits required fields, or uses the wrong types. Catch parsing/validation errors at the application boundary and decide whether to ask for a corrected response, return a safe error, or send the item for human review.

Do not silently convert invalid data into a plausible-looking value.

In [30]:
import json
from pydantic import ValidationError
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

bad_output = '{"course": "Python", "duration": }'

try:
    result = parser.parse(bad_output)
    print(result)
except Exception as exc:
    print("Parsing failed:", type(exc).__name__)
    print("Do not pass this invalid result to the next business step.")

Parsing failed: OutputParserException
Do not pass this invalid result to the next business step.


## 30. Chain debugging – inspect each stage

When a chain fails, test each stage independently:
1. Format the prompt and inspect the messages.
2. Invoke the model and inspect the raw response.
3. Pass the raw response to the parser.
4. Check the final output type and required fields.

This identifies whether the issue is prompt formatting, provider/model execution, or parsing.

In [31]:
debug_prompt = PromptTemplate.from_template(
    "Return a JSON object with keys topic and definition for: {topic}"
)

formatted_prompt = debug_prompt.format(topic="API")
print("1. FORMATTED PROMPT:")
print(formatted_prompt)

raw_response = llm.invoke(formatted_prompt)
print("\\n2. RAW MODEL RESPONSE:")
print(raw_response.content)

# Only parse if the model returned valid JSON.
try:
    parsed_response = JsonOutputParser().parse(raw_response.content)
    print("\\n3. PARSED RESPONSE:")
    print(parsed_response)
    print("4. OUTPUT TYPE:", type(parsed_response))
except Exception as exc:
    print("Parser stage failed:", type(exc).__name__)

1. FORMATTED PROMPT:
Return a JSON object with keys topic and definition for: API
\n2. RAW MODEL RESPONSE:
{
  "topic": "API",
  "definition": "An Application Programming Interface (API) is a set of rules, protocols, and tools that allows different software applications to communicate with each other. It defines the methods and data formats that developers can use to request services or data from a system, enabling integration, automation, and the building of complex functionalities without needing to understand the underlying codebase."
}
\n3. PARSED RESPONSE:
{'topic': 'API', 'definition': 'An Application Programming Interface (API) is a set of rules, protocols, and tools that allows different software applications to communicate with each other. It defines the methods and data formats that developers can use to request services or data from a system, enabling integration, automation, and the building of complex functionalities without needing to understand the underlying codebase.'}

## 31. LCEL mini-project – IT support ticket workflow

**Goal:** Accept a ticket, classify it, generate a short action, and return a combined result.

**Workflow:**
1. Input ticket text.
2. Parallel branches create a summary and category.
3. A final prompt recommends the next action.
4. Return all fields as a dictionary.

For a production system, do not rely on free-form categories for access control or ticket automation. Validate against an allowlist and require human approval for high-impact actions.

In [32]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

summary_branch = (
    PromptTemplate.from_template("Summarize this ticket in one sentence: {ticket}")
    | llm | StrOutputParser()
)

category_branch = (
    PromptTemplate.from_template(
        "Classify this ticket as Technical, Billing, or Account. Return only one category: {ticket}"
    )
    | llm | StrOutputParser()
)

analysis = RunnableParallel(
    summary=summary_branch,
    category=category_branch
)

action_chain = (
    PromptTemplate.from_template(
        "For a {category} ticket with this summary: {summary}. "
        "Suggest one safe next step for a support agent."
    )
    | llm | StrOutputParser()
)

workflow = (
    RunnablePassthrough.assign(
        analysis=analysis
    )
    | RunnablePassthrough.assign(
        next_action=lambda x: action_chain.invoke({
            "category": x["analysis"]["category"],
            "summary": x["analysis"]["summary"]
        })
    )
)

workflow_result = workflow.invoke({
    "ticket": "I cannot access my account after changing my password."
})

print(workflow_result)

{'ticket': 'I cannot access my account after changing my password.', 'analysis': {'summary': 'User reports being locked out of their account after a password change.', 'category': 'Account'}, 'next_action': '**Safe next step:** Verify the user’s identity (e.g., by asking for a pre‑registered secondary email, phone number, or security question) and then check the account’s lockout status in the admin console. If the account is indeed locked, follow your organization’s standard procedure to unlock it and, if needed, issue a temporary password or guide the user through a secure password reset. This ensures the user’s identity is confirmed before any changes are made to the account.'}


## 32. Exercises and interview questions

### Hands-on exercises
1. Create a `CommaSeparatedListOutputParser` chain that returns five AWS services.
2. Create a Pydantic model for a cloud incident with `service`, `severity`, `impact`, and `recommended_action`.
3. Use `RunnableParallel` to generate a definition, use case, and quiz from one topic.
4. Use `RunnablePassthrough.assign()` to preserve the original input and add a generated summary.
5. Use `RunnableBranch` to route Beginner and Advanced learners to different prompts.
6. Use `with_fallbacks()` to provide a safe response when a mock runnable fails.
7. Use `batch()` to process five course topics.
8. Debug a malformed JSON response by inspecting the raw output before parsing.

### Interview questions
- What is LCEL, and what does the pipe operator do?
- What is the difference between `RunnableSequence` and `RunnableParallel`?
- What is the difference between `RunnablePassthrough` and `RunnableLambda`?
- When would you choose `JsonOutputParser` over `PydanticOutputParser`?
- What is the difference between `parse()` and `invoke()`?
- Why can a valid JSON response still be incorrect for a business application?
- What are the trade-offs of batch, streaming, and retries?
- How do you debug a chain that returns malformed output?

### Completion target
You should be able to explain the data type entering and leaving each stage of a chain, select a parser for a use case, and compose at least three runnable components into a working workflow.